In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')
print(sys.path)

['/home/hieutt/CAN-SupCon-IDS/notebooks', '/home/hieutt/miniconda3/envs/torchtf/lib/python39.zip', '/home/hieutt/miniconda3/envs/torchtf/lib/python3.9', '/home/hieutt/miniconda3/envs/torchtf/lib/python3.9/lib-dynload', '', '/home/hieutt/miniconda3/envs/torchtf/lib/python3.9/site-packages', '../']


## Import và path

In [2]:
import os
import json
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from dataset_sequence_graph import (
    SequenceGraphDataset,
    collate_sequence_graphs,
    move_sequence_batch_to_device,
)

# dùng model mới của bạn
from networks.graph_attention_encoder import GraphAttentionEncoder
from networks.graph_temporal_kan_v3 import GraphTemporalKANv3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEQ_FOLDER = "../data/2017-subaru-forester/seq_temporal_majority"
CKPT_PATH = "../save/graph_temporal_kan_v4_auxattn_temporal_majority/graph_temporal_kan_v4_auxattn_temporal_majority_best_macro_f1.pth"
OUT_JSONL = "../save/graph_temporal_kan_v4_auxattn_temporal_majority/test_evidence.jsonl"
BATCH_SIZE = 64

## Load dataset

In [3]:
test_dataset = SequenceGraphDataset(
    seq_path=str(Path(SEQ_FOLDER) / "test_seq.pt"),
    return_meta=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_sequence_graphs,
)

label_mapping = test_dataset.label_mapping
id_to_label = {int(k): v for k, v in label_mapping.items()}
label_to_id = {v: k for k, v in id_to_label.items()}

print("num test sequences:", len(test_dataset))
print("seq_len:", test_dataset.seq_len)
print("label_mapping:", id_to_label)
print("node_feat_dim:", test_dataset.infer_node_feat_dim())
print("edge_attr_dim:", test_dataset.infer_edge_attr_dim())
print("num_ids:", test_dataset.infer_num_ids())

num test sequences: 55570
seq_len: 3
label_mapping: {0: 'normal', 1: 'combined', 2: 'dos', 3: 'fuzzing', 4: 'normal', 5: 'normal', 6: 'rpm', 7: 'speed', 8: 'standstill', 9: 'systematic'}
node_feat_dim: 16
edge_attr_dim: 6
num_ids: 1568


## Watch one sample

In [4]:
sample = test_dataset[0]
print("sequence_id:", sample["sequence_id"])
print("y:", sample["y"].item(), "->", id_to_label[int(sample["y"].item())])
print("graph_labels:", sample["graph_labels"])
print("meta:", sample["meta"])
print("num graphs:", len(sample["graphs"]))
print("graph[0]:", sample["graphs"][0])

sequence_id: test__combined__run00000001__chunk00000000__seq00000000
y: 0 -> normal
graph_labels: ['normal', 'normal', 'normal']
meta: {'source_class': 'combined', 'seq_len': 3, 'graph_run_id': 1, 'chunk_id': 'combined__run00000001__chunk00000000', 'chunk_label': 'normal', 'chunk_num_graphs': 16, 'start_msg_idx_in_file': 0, 'end_msg_idx_in_file': 1023}
num graphs: 3
graph[0]: Data(x=[64, 16], edge_index=[2, 315], edge_attr=[315, 6], y=[1], edge_type=[315], id_token=[64], arbitration_id=[64], timestamp=[64], msg_idx_in_file=[64], attack_count=[1], attack_ratio=[1], is_mixed_window=[1], has_attack=[1])


## Rebuild model from checkpoint args

In [5]:
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
args = ckpt["args"]

print(json.dumps(args, indent=2))

{
  "sequence_folder": "./data/2017-subaru-forester/seq_temporal_majority",
  "save_folder": "./save/graph_temporal_kan_v4_auxattn_temporal_majority",
  "model_name": "graph_temporal_kan_v4_auxattn_temporal_majority",
  "graph_encoder_ckpt": "",
  "strict_graph_encoder_load": false,
  "freeze_graph_encoder_epochs": 0,
  "batch_size": 64,
  "num_workers": 8,
  "pin_memory": false,
  "max_train_sequences": 0,
  "max_val_sequences": 0,
  "max_test_sequences": 0,
  "subsample_train_frac": 0.2,
  "subsample_val_frac": 0.2,
  "subsample_test_frac": 0.2,
  "subsample_min_per_class": 0,
  "subsample_seed": 42,
  "epochs": 100,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "grad_clip": 1.0,
  "print_freq": 50,
  "device": "cuda",
  "seed": 42,
  "amp": false,
  "resume": "",
  "scheduler": "cosine",
  "t0": 10,
  "tmult": 2,
  "eta_min": 1e-06,
  "use_weighted_sampler": false,
  "use_class_weights": false,
  "loss_name": "ce",
  "label_smoothing": 0.0,
  "focal_gamma": 2.0,
  "ldam_max_

## Build encoder + temporal model

In [6]:
node_feat_dim = test_dataset.infer_node_feat_dim()
edge_attr_dim = test_dataset.infer_edge_attr_dim()
num_ids = test_dataset.infer_num_ids()
num_classes = len(id_to_label)
seq_len = test_dataset.seq_len

graph_encoder = GraphAttentionEncoder(
    node_feat_dim=node_feat_dim,
    edge_attr_dim=edge_attr_dim,
    num_ids=num_ids,
    hidden_dim=args["hidden_dim"],
    num_layers=args["num_layers"],
    heads=args["heads"],
    id_emb_dim=args["id_emb_dim"],
    rel_emb_dim=args["rel_emb_dim"],
    num_relations=args["num_relations"],
    dropout=args["dropout"],
    ffn_ratio=args["ffn_ratio"],
)

graph_emb_dim = args["hidden_dim"] * 2

model = GraphTemporalKANv3(
    graph_encoder=graph_encoder,
    num_classes=num_classes,
    seq_len=seq_len,
    graph_emb_dim=graph_emb_dim,
    temporal_hidden_dim=args["temporal_hidden_dim"],
    num_transformer_layers=args["num_transformer_layers"],
    num_transformer_heads=args["num_transformer_heads"],
    dropout=args["dropout"],
    kan_hidden=args["kan_hidden"],
    kan_grid_size=args["kan_grid_size"],
    kan_spline_order=args["kan_spline_order"],
    use_cls_token=args.get("use_cls_token", True),
).to(DEVICE)

model.load_state_dict(ckpt["model"], strict=True)
model.eval()

print("loaded checkpoint epoch:", ckpt.get("epoch", "N/A"))

/home/hieutt/miniconda3/envs/torchtf/lib/python3.9/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


loaded checkpoint epoch: 74


## Helper functions for extract evidence

In [8]:
def softmax_np(x):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x)
    e = np.exp(x)
    return e / np.sum(e)

def get_topk_probs(logits_row, k=3):
    probs = torch.softmax(logits_row, dim=-1).detach().cpu().numpy()
    top_idx = np.argsort(probs)[::-1][:k]
    return [
        {
            "class_id": int(i),
            "label": id_to_label[int(i)],
            "prob": float(probs[i]),
        }
        for i in top_idx
    ]

def get_global_relation_gates(graph_encoder):
    # relation gate ở từng block: softmax(rel_gate)
    out = []
    relation_names = ["temporal", "same_id", "payload_similarity", "timing_affinity"]
    for b_idx, block in enumerate(graph_encoder.blocks):
        weights = torch.softmax(block.rel_gate.detach().cpu(), dim=0).numpy()
        out.append({
            "block_index": int(b_idx),
            "weights": {
                relation_names[r]: float(weights[r]) for r in range(len(weights))
            }
        })
    return out

## Test with one batch

In [9]:
batch = next(iter(test_loader))
batch = move_sequence_batch_to_device(batch, DEVICE)

with torch.no_grad():
    logits, aux = model(batch["graph_batches"], update_grid=False, return_aux=True)

print("logits shape:", logits.shape)
print("aux keys:", list(aux.keys()))
for k, v in aux.items():
    if torch.is_tensor(v):
        print(k, tuple(v.shape))

logits shape: torch.Size([64, 10])
aux keys: ['token_graphs', 'last_graph_embedding', 'aux_last_logits', 'cls_token', 'attn_pool', 'mean_pool', 'last_token', 'attn_weights', 'fused']
token_graphs (64, 3, 256)
last_graph_embedding (64, 256)
aux_last_logits (64, 10)
cls_token (64, 256)
attn_pool (64, 256)
mean_pool (64, 256)
last_token (64, 256)
attn_weights (64, 3)
fused (64, 256)


## Extract evidence package for whole test set

In [10]:
def build_evidence_records(model, loader, device, max_batches=None):
    model.eval()
    records = []

    global_rel_gates = get_global_relation_gates(model.graph_encoder)

    with torch.no_grad():
        for b_idx, batch in enumerate(loader):
            if max_batches is not None and b_idx >= max_batches:
                break

            batch = move_sequence_batch_to_device(batch, device)

            logits, aux = model(batch["graph_batches"], update_grid=False, return_aux=True)
            probs = torch.softmax(logits, dim=-1)
            preds = torch.argmax(probs, dim=-1)

            attn = aux["attn_weights"].detach().cpu().numpy()   # [B, L]
            logits_cpu = logits.detach().cpu()
            preds_cpu = preds.detach().cpu().numpy()
            y_cpu = batch["y"].detach().cpu().numpy()

            sequence_ids = batch.get("sequence_ids", [f"seq_{b_idx}_{i}" for i in range(len(y_cpu))])
            graph_ids = batch.get("graph_ids", [[] for _ in range(len(y_cpu))])
            graph_labels = batch.get("graph_labels", [[] for _ in range(len(y_cpu))])
            metas = batch.get("meta", [{} for _ in range(len(y_cpu))])

            for i in range(len(y_cpu)):
                attn_i = attn[i]
                top_t = int(np.argmax(attn_i))

                record = {
                    "sequence_id": sequence_ids[i],
                    "true_class_id": int(y_cpu[i]),
                    "true_label": id_to_label[int(y_cpu[i])],
                    "pred_class_id": int(preds_cpu[i]),
                    "pred_label": id_to_label[int(preds_cpu[i])],
                    "correct": bool(int(y_cpu[i]) == int(preds_cpu[i])),
                    "top3_probs": get_topk_probs(logits_cpu[i], k=3),
                    "confidence": float(torch.max(probs[i]).item()),
                    "attention_weights": [float(x) for x in attn_i.tolist()],
                    "most_important_timestep": top_t,
                    "most_important_graph_label": graph_labels[i][top_t] if len(graph_labels[i]) > top_t else None,
                    "graph_labels": graph_labels[i],
                    "graph_ids": graph_ids[i],
                    "meta": metas[i],
                    "global_relation_gates": global_rel_gates,
                }

                # có thể thêm aux classifier nếu model trả ra
                if "last_graph_logits" in aux:
                    last_probs = torch.softmax(aux["last_graph_logits"][i], dim=-1).detach().cpu().numpy()
                    last_top = int(np.argmax(last_probs))
                    record["aux_last_prediction"] = {
                        "class_id": last_top,
                        "label": id_to_label[last_top],
                        "confidence": float(np.max(last_probs)),
                    }

                records.append(record)

    return records

In [11]:
records = build_evidence_records(model, test_loader, DEVICE)
print("num records:", len(records))
print(json.dumps(records[0], indent=2, ensure_ascii=False))

num records: 55570
{
  "sequence_id": "test__combined__run00000001__chunk00000000__seq00000000",
  "true_class_id": 0,
  "true_label": "normal",
  "pred_class_id": 0,
  "pred_label": "normal",
  "correct": true,
  "top3_probs": [
    {
      "class_id": 0,
      "label": "normal",
      "prob": 0.9999656677246094
    },
    {
      "class_id": 1,
      "label": "combined",
      "prob": 3.426914190640673e-05
    },
    {
      "class_id": 2,
      "label": "dos",
      "prob": 1.0618220613878293e-07
    }
  ],
  "confidence": 0.9999655485153198,
  "attention_weights": [
    0.9067659378051758,
    0.033020760864019394,
    0.060213346034288406
  ],
  "most_important_timestep": 0,
  "most_important_graph_label": "normal",
  "graph_labels": [
    "normal",
    "normal",
    "normal"
  ],
  "graph_ids": [
    "combined__00000000_00000064",
    "combined__00000064_00000128",
    "combined__00000128_00000192"
  ],
  "meta": {
    "source_class": "combined",
    "seq_len": 3,
    "graph_run_

## Save JSONL for LLM

In [12]:
os.makedirs(Path(OUT_JSONL).parent, exist_ok=True)

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("saved:", OUT_JSONL)

saved: ../save/graph_temporal_kan_v4_auxattn_temporal_majority/test_evidence.jsonl


## Basic prompt template 

In [13]:
def build_llm_prompt(record):
    return f"""
Bạn là chuyên gia an ninh IVN/CAN.
Hãy giải thích quyết định của mô hình CHỈ dựa trên bằng chứng bên dưới.
Không được suy đoán ngoài dữ liệu. Nếu bằng chứng chưa đủ, hãy nói rõ là chưa đủ.

[Prediction summary]
- Sequence ID: {record['sequence_id']}
- True label: {record['true_label']}
- Predicted label: {record['pred_label']}
- Correct: {record['correct']}
- Confidence: {record['confidence']:.4f}

[Top-3 class probabilities]
{json.dumps(record['top3_probs'], ensure_ascii=False, indent=2)}

[Temporal evidence]
- Graph labels in sequence: {record['graph_labels']}
- Attention weights: {record['attention_weights']}
- Most important timestep: {record['most_important_timestep']}
- Most important graph label: {record['most_important_graph_label']}

[Graph-relation evidence]
{json.dumps(record['global_relation_gates'], ensure_ascii=False, indent=2)}

[Task]
Hãy trả lời theo 4 mục:
1. Tóm tắt ngắn gọn vì sao mô hình dự đoán nhãn này.
2. Timestep nào quan trọng nhất và điều đó gợi ý gì.
3. Có dấu hiệu nào cho thấy dự đoán có thể không chắc chắn hoặc có thể nhầm sang class khác không.
4. Viết một cảnh báo ngắn gọn 2-3 câu cho analyst.
""".strip()

In [14]:
example_prompt = build_llm_prompt(records[0])
print(example_prompt[:3000])

Bạn là chuyên gia an ninh IVN/CAN.
Hãy giải thích quyết định của mô hình CHỈ dựa trên bằng chứng bên dưới.
Không được suy đoán ngoài dữ liệu. Nếu bằng chứng chưa đủ, hãy nói rõ là chưa đủ.

[Prediction summary]
- Sequence ID: test__combined__run00000001__chunk00000000__seq00000000
- True label: normal
- Predicted label: normal
- Correct: True
- Confidence: 1.0000

[Top-3 class probabilities]
[
  {
    "class_id": 0,
    "label": "normal",
    "prob": 0.9999656677246094
  },
  {
    "class_id": 1,
    "label": "combined",
    "prob": 3.426914190640673e-05
  },
  {
    "class_id": 2,
    "label": "dos",
    "prob": 1.0618220613878293e-07
  }
]

[Temporal evidence]
- Graph labels in sequence: ['normal', 'normal', 'normal']
- Attention weights: [0.9067659378051758, 0.033020760864019394, 0.060213346034288406]
- Most important timestep: 0
- Most important graph label: normal

[Graph-relation evidence]
[
  {
    "block_index": 0,
    "weights": {
      "temporal": 0.2664647400379181,
      "s

## Init client and helper for prompt

In [19]:
from openai import OpenAI
import json
from pathlib import Path
from typing import Dict, Any, List

# ====== config ======
OLLAMA_MODEL = "qwen3:8b"
OLLAMA_BASE_URL = "http://localhost:11434/v1/"
LLM_OUT_JSONL = "./save/graph_temporal_kan_v3_auxattn/test_llm_explanations_ollama_qwen3_8b.jsonl"

# Ollama dùng OpenAI-compatible API local
client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama",   # required by client, but ignored by Ollama
)

def build_llm_prompt(record):
    return f"""
Bạn là chuyên gia an ninh IVN/CAN, làm nhiệm vụ giải thích hậu kiểm cho một mô hình IDS.

Bạn PHẢI tuân thủ các quy tắc sau:
1. Chỉ sử dụng thông tin xuất hiện trực tiếp trong input.
2. Không được suy đoán thêm về nguyên nhân, bối cảnh, hay kiểu tấn công ngoài evidence đã cho.
3. Nếu một bằng chứng là global model evidence thay vì sample-specific evidence, phải nói rõ điều đó.
4. Nếu bằng chứng chưa đủ để kết luận mạnh, phải nói rõ là "chưa đủ bằng chứng".
5. Không đưa ra lời khuyên chung chung hoặc sáo rỗng.
6. Cảnh báo analyst phải ngắn, cụ thể, bám sát sample hiện tại.
7. Chỉ trả về JSON hợp lệ, không dùng markdown, không thêm text ngoài JSON.

[Prediction summary]
- Sequence ID: {record['sequence_id']}
- True label: {record['true_label']}
- Predicted label: {record['pred_label']}
- Correct: {record['correct']}
- Confidence: {record['confidence']:.6f}

[Top-3 class probabilities]
{json.dumps(record['top3_probs'], ensure_ascii=False, indent=2)}

[Temporal evidence]
- Graph labels in sequence: {record['graph_labels']}
- Attention weights: {record['attention_weights']}
- Most important timestep: {record['most_important_timestep']}
- Most important graph label: {record['most_important_graph_label']}

[Graph-relation evidence]
{json.dumps(record['global_relation_gates'], ensure_ascii=False, indent=2)}

[Required JSON format]
{{
  "summary": "Một câu ngắn giải thích dự đoán chính.",
  "important_timestep_analysis": "Phân tích timestep quan trọng nhất, chỉ dựa trên attention và graph label tương ứng.",
  "uncertainty_or_confusion": "Nêu dấu hiệu chắc chắn/không chắc chắn, và class dễ nhầm nếu có bằng chứng từ top probabilities.",
  "analyst_warning": "Cảnh báo ngắn 1-2 câu, cụ thể cho analyst.",
  "evidence_used": [
    "Mỗi dòng là một bằng chứng ngắn, trực tiếp từ input"
  ],
  "evidence_scope_note": "Nêu rõ nếu có dùng bằng chứng global model-level."
}}

[Additional constraints]
- Nếu confidence >= 0.95 và top-2 thấp hơn rõ rệt, hãy nói rõ đây là high-confidence prediction.
- Nếu predicted label khác true label, phải nhắc rõ đây là misclassification.
- Không được viết các câu kiểu: 'hãy kiểm tra thêm nếu nghi ngờ', trừ khi input thực sự cho thấy uncertainty.
""".strip()

def select_records_for_explanation(records: List[Dict[str, Any]], max_items=20):
    hard_labels = {"gear", "rpm"}
    selected = []

    high_conf_correct = [r for r in records if r["correct"] and r["confidence"] >= 0.95]
    wrong_cases = [r for r in records if not r["correct"]]
    hard_cases = [r for r in records if r["true_label"] in hard_labels or r["pred_label"] in hard_labels]

    selected.extend(high_conf_correct[: max_items // 3])
    selected.extend(wrong_cases[: max_items // 3])
    selected.extend(hard_cases[: max_items - len(selected)])

    uniq = {}
    for r in selected:
        uniq[r["sequence_id"]] = r

    return list(uniq.values())

sample_records = select_records_for_explanation(records, max_items=12)
print("Số record sẽ thử trước:", len(sample_records))
print(sample_records[0]["sequence_id"], sample_records[0]["true_label"], sample_records[0]["pred_label"])

Số record sẽ thử trước: 12
test__combined__run00000001__chunk00000000__seq00000000 normal normal


## Explain one sample with LLM

In [20]:
def extract_json_from_text(text: str) -> Dict[str, Any]:
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = text[start:end+1]
        return json.loads(candidate)

    raise ValueError("Không parse được JSON từ output của LLM.")

def explain_one_record_ollama(record: Dict[str, Any], model_name: str = OLLAMA_MODEL) -> Dict[str, Any]:
    prompt = build_llm_prompt(record)

    response = client.responses.create(
        model=model_name,
        input=prompt,
    )

    raw_text = response.output_text
    parsed = extract_json_from_text(raw_text)

    return {
        "sequence_id": record["sequence_id"],
        "true_label": record["true_label"],
        "pred_label": record["pred_label"],
        "correct": record["correct"],
        "confidence": record["confidence"],
        "llm_model": model_name,
        "llm_explanation": parsed,
        "raw_llm_text": raw_text,
    }

# thử với 1 mẫu
one_result = explain_one_record_ollama(sample_records[0])
print(json.dumps(one_result, ensure_ascii=False, indent=2))

{
  "sequence_id": "test__combined__run00000001__chunk00000000__seq00000000",
  "true_label": "normal",
  "pred_label": "normal",
  "correct": true,
  "confidence": 0.9999655485153198,
  "llm_model": "qwen3:8b",
  "llm_explanation": {
    "summary": "Mô hình xác định dòng mạng là normal với độ tin cậy cao, không có dấu hiệu tấn công.",
    "important_timestep_analysis": "Timestep quan trọng nhất (0) có nhãn 'normal' và trọng số attention 0.9068, chiếm ưu thế trong chuỗi.",
    "uncertainty_or_confusion": "Không có sự phân mảnh đáng kể trong top-3 class probabilities (normal: ~100%, combined/dos: <0.0001%).",
    "analyst_warning": "Dữ liệu không chứa bất kỳ dấu hiệu khả nghi nào, nhưng cần kiểm tra thêm nếu có cảnh báo từ hệ thống khác.",
    "evidence_used": [
      "Sequence ID: test__combined__run00000001__chunk00000000__seq00000000",
      "True label: normal, Predicted label: normal, Correct: True, Confidence: 0.999966",
      "Graph labels in sequence: ['normal', 'normal', 'norma

## Test with three cases

In [21]:
def pick_three_case_types(records, hard_labels=("gear", "rpm")):
    # 1) đúng nhưng confidence thấp
    correct_cases = [r for r in records if r["correct"]]
    low_conf_correct = None
    if correct_cases:
        low_conf_correct = sorted(correct_cases, key=lambda x: x["confidence"])[0]

    # 2) case sai
    wrong_cases = [r for r in records if not r["correct"]]
    wrong_case = None
    if wrong_cases:
        wrong_case = sorted(wrong_cases, key=lambda x: x["confidence"], reverse=True)[0]

    # 3) case class khó
    hard_cases = [
        r for r in records
        if (r["true_label"] in hard_labels) or (r["pred_label"] in hard_labels)
    ]
    hard_case = None
    if hard_cases:
        # ưu tiên case sai hoặc confidence thấp trong nhóm hard
        hard_case = sorted(
            hard_cases,
            key=lambda x: (x["correct"], x["confidence"])
        )[0]

    selected = {
        "low_conf_correct": low_conf_correct,
        "wrong_case": wrong_case,
        "hard_case": hard_case,
    }
    return selected

In [22]:
selected_cases = pick_three_case_types(records)

for name, rec in selected_cases.items():
    print("=" * 100)
    print("CASE TYPE:", name)
    if rec is None:
        print("Không tìm thấy case phù hợp.")
        continue
    print("sequence_id :", rec["sequence_id"])
    print("true_label  :", rec["true_label"])
    print("pred_label  :", rec["pred_label"])
    print("correct     :", rec["correct"])
    print("confidence  :", rec["confidence"])
    print("top3_probs  :", rec["top3_probs"])
    print("graph_labels:", rec["graph_labels"])
    print("attn        :", rec["attention_weights"])

CASE TYPE: low_conf_correct
sequence_id : test__speed__run00000001__chunk00000186__seq00000011
true_label  : normal
pred_label  : normal
correct     : True
confidence  : 0.5238505601882935
top3_probs  : [{'class_id': 0, 'label': 'normal', 'prob': 0.5238505601882935}, {'class_id': 7, 'label': 'speed', 'prob': 0.4759766757488251}, {'class_id': 1, 'label': 'combined', 'prob': 0.00011124659067718312}]
graph_labels: ['normal', 'normal', 'speed']
attn        : [1.9372858872657162e-10, 0.0, 1.0]
CASE TYPE: wrong_case
sequence_id : test__combined__run00000001__chunk00000936__seq00000002
true_label  : combined
pred_label  : normal
correct     : False
confidence  : 0.9999997615814209
top3_probs  : [{'class_id': 0, 'label': 'normal', 'prob': 0.9999997615814209}, {'class_id': 1, 'label': 'combined', 'prob': 1.7057000434306246e-07}, {'class_id': 8, 'label': 'standstill', 'prob': 9.520960020381608e-08}]
graph_labels: ['combined', 'combined', 'normal']
attn        : [0.062055137008428574, 0.878218352

In [23]:
def extract_json_from_text(text):
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        return json.loads(text[start:end+1])

    raise ValueError("Không parse được JSON từ output của model.")

def explain_one_record_ollama(record, model_name="qwen3:8b"):
    prompt = build_llm_prompt(record)

    response = client.responses.create(
        model=model_name,
        input=prompt,
    )

    raw_text = response.output_text
    parsed = extract_json_from_text(raw_text)

    return {
        "sequence_id": record["sequence_id"],
        "true_label": record["true_label"],
        "pred_label": record["pred_label"],
        "correct": record["correct"],
        "confidence": record["confidence"],
        "llm_model": model_name,
        "llm_explanation": parsed,
        "raw_llm_text": raw_text,
    }

In [24]:
three_case_outputs = {}

for name, rec in selected_cases.items():
    if rec is None:
        three_case_outputs[name] = {"error": "No matching record found"}
        continue

    try:
        out = explain_one_record_ollama(rec, model_name="qwen3:8b")
        three_case_outputs[name] = out
        print(f"[OK] {name} -> {rec['sequence_id']}")
    except Exception as e:
        three_case_outputs[name] = {
            "sequence_id": rec["sequence_id"],
            "error": str(e),
        }
        print(f"[ERROR] {name} -> {rec['sequence_id']} -> {e}")

[OK] low_conf_correct -> test__speed__run00000001__chunk00000186__seq00000011
[OK] wrong_case -> test__combined__run00000001__chunk00000936__seq00000002
[OK] hard_case -> test__rpm__run00000001__chunk00001493__seq00000012


In [25]:
for name, out in three_case_outputs.items():
    print("\n" + "#" * 120)
    print("CASE TYPE:", name)
    print("#" * 120)
    print(json.dumps(out, ensure_ascii=False, indent=2))


########################################################################################################################
CASE TYPE: low_conf_correct
########################################################################################################################
{
  "sequence_id": "test__speed__run00000001__chunk00000186__seq00000011",
  "true_label": "normal",
  "pred_label": "normal",
  "correct": true,
  "confidence": 0.5238505601882935,
  "llm_model": "qwen3:8b",
  "llm_explanation": {
    "summary": "Mô hình dự đoán 'normal' với độ tin cậy thấp, có dấu hiệu nhầm lẫn giữa lớp 'normal' và 'speed'.",
    "important_timestep_analysis": "Timestep quan trọng nhất (2) có nhãn 'speed' với trọng số attention 1.0, nhưng nhãn thật của sequence là 'normal'.",
    "uncertainty_or_confusion": "Độ tin cậy 0.52 thấp, lớp 'speed' (47.6%) gần bằng 'normal' (52.4%) gây nhầm lẫn.",
    "analyst_warning": "Kiểm tra liệu timestep 2 có chứa hành vi 'speed' hợp lệ hay là false positive.",
    "ev

## Run batch explainations & save JSONL

In [ ]:
def run_llm_batch_ollama(records_to_explain: List[Dict[str, Any]], model_name: str = OLLAMA_MODEL):
    outputs = []

    for idx, rec in enumerate(records_to_explain, start=1):
        try:
            out = explain_one_record_ollama(rec, model_name=model_name)
            outputs.append(out)
            print(f"[{idx}/{len(records_to_explain)}] OK - {rec['sequence_id']}")
        except Exception as e:
            outputs.append({
                "sequence_id": rec["sequence_id"],
                "true_label": rec["true_label"],
                "pred_label": rec["pred_label"],
                "correct": rec["correct"],
                "confidence": rec["confidence"],
                "llm_model": model_name,
                "llm_explanation": None,
                "raw_llm_text": None,
                "error": str(e),
            })
            print(f"[{idx}/{len(records_to_explain)}] ERROR - {rec['sequence_id']} - {e}")

    return outputs

llm_results = run_llm_batch_ollama(sample_records, model_name=OLLAMA_MODEL)

Path(LLM_OUT_JSONL).parent.mkdir(parents=True, exist_ok=True)
with open(LLM_OUT_JSONL, "w", encoding="utf-8") as f:
    for item in llm_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Đã lưu:", LLM_OUT_JSONL)
print("Số kết quả:", len(llm_results))